<a href="https://colab.research.google.com/github/Samyuktha-P0/Secure-Digital-Document-Management-System/blob/main/SDDMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Install Required Libraries

In [15]:
!apt-get install tesseract-ocr -y

!pip install pytesseract
!pip install PyMuPDF
!pip install python-pptx
!pip install python-docx
!pip install pandas
!pip install pillow
!pip install langdetect

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 11 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.0 MB/s eta 0:00:00


Step 2: Import Libraries

In [24]:
import os
import re
import fitz
import pandas as pd
import pytesseract

from PIL import Image
from google.colab import files
from pptx import Presentation
from docx import Document
from langdetect import detect
from datetime import datetime

Step 3: Upload Document

In [5]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded File:", file_name)

Saving SDDMS_Presentation.pptx to SDDMS_Presentation.pptx
Uploaded File: SDDMS_Presentation.pptx


Step 4: File Validation

In [19]:
import os

# Supported file types
ALLOWED_EXTENSIONS = [
    '.pdf',
    '.png',
    '.jpg',
    '.jpeg',
    '.pptx',
    '.docx',
    '.txt',
    '.csv',
    '.xlsx'
]

# Maximum file size (MB)
MAX_FILE_SIZE_MB = 20

def validate_file(file_path):

    # File extension
    ext = os.path.splitext(file_path)[1].lower()

    # File size
    size_mb = os.path.getsize(file_path) / (1024 * 1024)

    print("===== FILE VALIDATION REPORT =====")
    print("File Name      :", os.path.basename(file_path))
    print("File Extension :", ext)
    print("File Size      :", round(size_mb, 2), "MB")

    # Extension validation
    if ext in ALLOWED_EXTENSIONS:
        print("File Type      : VALID")
    else:
        print("File Type      : INVALID")
        print("Supported Types:", ", ".join(ALLOWED_EXTENSIONS))
        return False

    # Size validation
    if size_mb <= MAX_FILE_SIZE_MB:
        print("File Size      : VALID")
    else:
        print("File Size      : INVALID")
        print(f"Maximum Allowed: {MAX_FILE_SIZE_MB} MB")
        return False

    print("Validation Status : SUCCESS")
    return True


# Example
validate_file(file_name)

===== FILE VALIDATION REPORT =====
File Name      : SDDMS_Presentation.pptx
File Extension : .pptx
File Size      : 0.11 MB
File Type      : VALID
File Size      : VALID
Validation Status : SUCCESS


True

Step 5: OCR Text Extraction

In [23]:
!pip install python-pptx
!pip install openpyxl

In [25]:
def extract_text(file_path):

    ext = os.path.splitext(file_path)[1].lower()

    extracted_text = ""

    try:

        # PDF
        if ext == ".pdf":

            pdf = fitz.open(file_path)

            for page in pdf:
                extracted_text += page.get_text() + "\n"

            pdf.close()

        # Images
        elif ext in [".png", ".jpg", ".jpeg"]:

            image = Image.open(file_path)

            extracted_text = pytesseract.image_to_string(image)

        # PPTX
        elif ext == ".pptx":

            prs = Presentation(file_path)

            for slide_num, slide in enumerate(prs.slides, start=1):

                extracted_text += f"\n--- Slide {slide_num} ---\n"

                for shape in slide.shapes:

                    if hasattr(shape, "text"):

                        extracted_text += shape.text + "\n"

        # DOCX
        elif ext == ".docx":

            doc = Document(file_path)

            for para in doc.paragraphs:

                extracted_text += para.text + "\n"

        # TXT
        elif ext == ".txt":

            with open(file_path,
                      "r",
                      encoding="utf-8",
                      errors="ignore") as file:

                extracted_text = file.read()

        # CSV
        elif ext == ".csv":

            df = pd.read_csv(file_path)

            extracted_text = df.to_string(index=False)

        # XLSX
        elif ext == ".xlsx":

            df = pd.read_excel(file_path)

            extracted_text = df.to_string(index=False)

        else:

            extracted_text = "Unsupported File Type"

    except Exception as e:

        extracted_text = f"Error: {e}"

    return extracted_text

In [26]:
text = extract_text(file_name)

In [27]:
print("===== EXTRACTED TEXT =====\n")

print(text)

print("\n=========================")

print("Characters :", len(text))
print("Words      :", len(text.split()))
print("Lines      :", len(text.splitlines()))

===== EXTRACTED TEXT =====


--- Slide 1 ---

SECURE DIGITAL DOCUMENT MANAGEMENT SYSTEM (SDDMS)
Project By
411424243074 – Samyuktha P
III Year – B Sec
Department of Artificial Intelligence and Data Science 
NPSBCET


Guided by
Mrs Sathya Priya
Designation
Department of Artificial Intelligence and Data Science 
NPSBCET
1
An OCR-Based Document Classification and Intelligent Retrieval System

--- Slide 2 ---

ABSTRACT
2
The Secure Digital Document Management System (SDDMS) is an intelligent platform for storing, organising, classifying and retrieving digital documents. Files uploaded in PDF, PNG, JPG or JPEG format are processed by an OCR engine that converts scanned pages and photographs into machine-readable text. The extracted content is analysed to identify keywords and to assign the document to a domain such as Education, Healthcare, Finance, Government or Legal. Document name, keywords, category, upload date and extracted text are stored as metadata, so that records can be located t

Step 6: Generate Statistics

In [28]:
word_count = len(extracted_text.split())

char_count = len(extracted_text)

line_count = len(extracted_text.splitlines())

print("Words:",word_count)
print("Characters:",char_count)
print("Lines:",line_count)

Words: 1334
Characters: 9025
Lines: 206


Step 7: Language Detection

In [29]:
try:
    language = detect(extracted_text)
except:
    language = "Unknown"

print("Language:",language)

Language: en


Step 8: Sensitive Data Detection

In [32]:
import re

# Emails
emails = re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', extracted_text)

# Phone Numbers
phones = re.findall(r'\b[6-9]\d{9}\b', extracted_text)


# CGPA
cgpa = re.findall(r'CGPA[:\s]*([0-9]+\.[0-9]+)', extracted_text, re.IGNORECASE)

# Department
departments = re.findall(
    r'Artificial Intelligence and Data Science|Computer Science|Information Technology|ECE|EEE|Mechanical',
    extracted_text,
    re.IGNORECASE
)

print("Emails:", emails if emails else "Not Found")
print("Phones:", phones if phones else "Not Found")
print("CGPA:", cgpa if cgpa else "Not Found")
print("Departments:", departments if departments else "Not Found")

Emails: Not Found
Phones: Not Found
CGPA: Not Found
Departments: ['Artificial Intelligence and Data Science', 'Artificial Intelligence and Data Science', 'EEE', 'EEE']


Step 9: Domain Classification

In [33]:
def classify_document(text):

    text = text.lower()

    if any(word in text for word in
           ['cgpa','semester','student','college']):
        return "Education"

    elif any(word in text for word in
             ['bank','account','loan']):
        return "Finance"

    elif any(word in text for word in
             ['hospital','doctor','patient']):
        return "Healthcare"

    elif any(word in text for word in
             ['aadhaar','passport','government']):
        return "Government"

    else:
        return "Others"

domain = classify_document(extracted_text)

print("Domain:",domain)

Domain: Education


Step 10: Metadata Extraction

In [34]:
metadata = {

    "Document_Name": file_name,

    "File_Type": ext,

    "File_Size_MB": round(size_mb,2),

    "Upload_Date":
    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "Word_Count": word_count,

    "Character_Count": char_count,

    "Language": language,

    "Domain": domain
}

metadata

{'Document_Name': 'SDDMS_Presentation.pptx',
 'File_Type': '.pptx',
 'File_Size_MB': 0.11,
 'Upload_Date': '2026-09-17 04:32:43',
 'Word_Count': 1334,
 'Character_Count': 9025,
 'Language': 'en',
 'Domain': 'Education'}

Step 11: Save Document Information

In [35]:
df = pd.DataFrame([metadata])

df.to_csv("document_metadata.csv",
          mode="a",
          index=False)

print("Metadata Saved")

Metadata Saved


Step 12: Keyword Extraction

In [36]:
words = extracted_text.split()

keywords = list(set(words))

print("Sample Keywords:")

print(keywords[:20])

Sample Keywords:
['At', 'processed', 'role-based', 'generated', 'Machine', 'Recognition,"', 'SIGKDD,', 'Name,', 'analysed', 'applies', 'pipeline', 'correct', '13', 'Interface', 'trained', 'effective', 'document', 'register', 'SYSTEM', 'against']


Step 13: Duplicate Document Detection

In [37]:
if os.path.exists("documents_text.csv"):

    old_df = pd.read_csv("documents_text.csv")

    if extracted_text in old_df["Text"].values:
        print("Duplicate Document Found")

else:
    print("No Previous Documents")

No Previous Documents


Step 14: Create Access Log

In [38]:
access_log = {

    "Document": file_name,

    "Access_Time":
    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

log_df = pd.DataFrame([access_log])

log_df.to_csv("access_log.csv",
              mode="a",
              index=False)

print("Access Logged")

Access Logged


Step 15: Password-Based Access

In [39]:
PASSWORD = "admin123"

user_password = input("Enter Password: ")

if user_password == PASSWORD:
    print("Access Granted")
else:
    print("Access Denied")

Enter Password: admin123
Access Granted


Step 16: Search Document

In [40]:
keyword = input("Enter Search Keyword: ")

if keyword.lower() in extracted_text.lower():
    print("Document Found")
else:
    print("No Match Found")

Enter Search Keyword: seach
No Match Found


Step 17: Document Summary

In [41]:
summary = " ".join(
    extracted_text.split()[:50]
)

print(summary)

SECURE DIGITAL DOCUMENT MANAGEMENT SYSTEM (SDDMS) Project By 411424243074 – Samyuktha P III Year – B Sec Department of Artificial Intelligence and Data Science NPSBCET Guided by Mrs Sathya Priya Designation Department of Artificial Intelligence and Data Science NPSBCET 1 An OCR-Based Document Classification and Intelligent Retrieval System ABSTRACT 2


Step 18: Analytics Dashboard

In [42]:
print("----- Dashboard -----")

print("Document Name:",file_name)

print("Domain:",domain)

print("Language:",language)

print("Words:",word_count)

print("Characters:",char_count)

print("File Size:",round(size_mb,2),"MB")

----- Dashboard -----
Document Name: SDDMS_Presentation.pptx
Domain: Education
Language: en
Words: 1334
Characters: 9025
File Size: 0.11 MB
